# LELA60331 Week 11 Seminar: Neural networks with PyTorch

Over the last few weeks we have written every step of training ourselves: the forward pass, the gradients and the weight updates. That is the best way to understand how training works, but it gets fiddly very quickly as models grow. In practice people use a library that does the fiddly parts for them. This week we learn **PyTorch**, one of the most widely used. It works out the **gradients** for us, provides ready-made **layers** and **optimisers**, and can run on a **GPU**.

| Part | Topic |
|---|---|
| 1 | Tensors and automatic gradients |
| 2 | Logistic regression with automatic gradients |
| 3 | Models, loss functions and optimisers |
| 4 | Using a GPU |
| 5 | A sentiment classifier in PyTorch |
| 6 | Bigger networks and dropout |

**Lectures:** this sheet builds on the Week 7 and Week 8 lectures on multilayer networks and how they are trained. Next week's sheet uses PyTorch to build the recurrent neural networks from this week's lecture.

---
# Part 1 · Tensors and automatic gradients

PyTorch's main type is the **tensor**. A tensor is very like a NumPy array: a list, table (or higher-dimensional block) of numbers, with element-by-element arithmetic. Neural networks usually use 32-bit numbers, `torch.float32`.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
print(a + b, a * 2, a.dot(b))
m = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(m.shape, m[1, 2], m.T.shape)
print(torch.tensor(np.array([1, 2, 3]), dtype=torch.float32))   # from a NumPy array
print(a.sum(), a.sum().item())          # .item() turns a one-number tensor into a Python number

### Automatic gradients

PyTorch's big advantage is that it can work out **gradients automatically**. If you create a tensor with `requires_grad=True`, PyTorch remembers every calculation that uses it. Calling `.backward()` on the final result then works out the gradient of that result with respect to the tensor, and stores it in the tensor's `.grad`.

For example, if `loss = (3w − 5)²`, then the gradient (how fast `loss` changes as `w` changes) is `2 × (3w − 5) × 3`. At `w = 2`, that is 2 × 1 × 3 = 6:

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss = (3 * w - 5) ** 2
loss.backward()
print(w.grad)

In Weeks 4 to 10 we worked out formulas for the gradients ourselves (such as `X.T.dot(q - y) / n`). With PyTorch, `backward()` does this for any calculation, however complicated.

### 🏋️ Exercise 1 — Automatic gradients

1. Create a tensor `v` holding 4.0 that requires gradients, calculate `result = v ** 2 + 2 * v`, and call `backward()`.
2. By hand, the gradient of v² + 2v is 2v + 2. Check that `v.grad` agrees.

In [ ]:
v = ???
result = ???
???
print(v.grad)        # 2 * 4 + 2 = 10

---
# Part 2 · Logistic regression with automatic gradients

Here is the simulated two-feature sentiment data from Week 4, with the extra column of 1s for the bias trick, converted to tensors.

In [ ]:
np.random.seed(10)
x = np.zeros((50, 2))
y = np.zeros(50)
for i in range(50):
    if np.random.random() > 0.5:
        x[i] = np.random.normal((2, 3))      # negative texts
    else:
        x[i] = np.random.normal((3, 2))      # positive texts
        y[i] = 1
x = np.column_stack([x, np.ones(50)])
x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)
print(x.shape, y.shape)

The training loop below is the logistic regression from Week 4, except that we never write the gradient formula: `loss.backward()` fills in `weights.grad`. Three PyTorch details:

* `torch.mv(x, weights)` is matrix-vector multiplication (like NumPy's `x.dot(weights)`), and `torch.sigmoid` is the sigmoid function.
* `torch.nn.BCELoss()` is PyTorch's **binary cross-entropy** loss, the same loss as before (averaged over the data points).
* The weight update must not itself be recorded for gradients, so it goes inside `with torch.no_grad():`. Afterwards we reset the gradient to `None`, because otherwise PyTorch would **add** the next gradient to the old one.

### 🏋️ Exercise 2 — Training with automatic gradients

Complete the loop: the forward pass (`z` and `q`), the loss, the backward pass, and the update.

In [ ]:
torch.manual_seed(0)
weights = torch.randn(3, requires_grad=True)
loss_fn = torch.nn.BCELoss()
n_iters = 5000
lr = 0.1
logistic_loss = []

for i in range(n_iters):
    z = ???
    q = ???
    loss = ???
    logistic_loss.append(loss.item())
    ???                     # work out weights.grad
    with torch.no_grad():
        weights -= ???
        weights.grad = None

plt.plot(logistic_loss)
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()
print(weights)

### Batches

As in Week 10, we usually train on **mini-batches**. The scikit-learn function `gen_batches(n, batch_size)` produces **slices** for consecutive batches: `x[s]` with a slice `s` selects that batch's rows. (For simplicity we don't shuffle here.)

In [ ]:
from sklearn.utils import gen_batches

for s in gen_batches(len(x), 20):
    print(s, x[s].shape)

---
# Part 3 · Models, loss functions and optimisers

PyTorch gives us two more tools that make training code short and easy to change.

### Models as classes

A PyTorch model is a **class** that inherits from `torch.nn.Module` (LELA60341 Week 7 covers classes and inheritance). It has:

* a constructor, `__init__`, which first calls the parent constructor with `super().__init__()` and then creates the model's **layers** as instance variables
* a method `forward(self, x)`, which does the forward pass: it takes a batch of inputs and returns the outputs

`nn.Linear(3, 1)` is a layer that takes 3 inputs and gives 1 output: a weighted sum, **including its own bias**. So we no longer need the extra column of 1s. Calling `model(inputs)` runs `forward`.

In [ ]:
import torch.nn as nn

class LogisticRegression(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.layer_1 = nn.Linear(num_features, 1)

    def forward(self, x):
        return torch.sigmoid(self.layer_1(x))

x2 = x[:, :2]                        # just the two features: nn.Linear has its own bias
model = LogisticRegression(2)
print(model)
print(model(x2[:3]))                 # outputs for the first 3 texts, before training

### Optimisers

An **optimiser** updates the weights for us. `torch.optim.SGD(model.parameters(), lr=0.1)` does exactly the gradient descent update we have been writing by hand (SGD stands for *stochastic gradient descent*, gradient descent on batches). Each training step is then:

1. `optimizer.zero_grad()`: reset the gradients
2. `pred = model(inputs)`: the forward pass
3. `loss = loss_fn(...)`, then `loss.backward()`: the loss and its gradients
4. `optimizer.step()`: update all the weights

### 🏋️ Exercise 3 — A complete PyTorch training loop

Complete the four steps. `model(inputs)` returns one column of outputs, so we use `.squeeze(1)` to turn it into a plain vector to compare with the labels.

In [ ]:
torch.manual_seed(0)
model = LogisticRegression(2)
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
n_epochs = 1000
epoch_loss = []

for epoch in range(n_epochs):
    total = 0.0
    for s in gen_batches(len(x2), 10):
        inputs, labels = x2[s], y[s]
        ???
        pred = ???
        loss = ???
        ???
        ???
        total += loss.item()
    epoch_loss.append(total)

plt.plot(epoch_loss)
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()
accuracy = ((model(x2).squeeze(1) > 0.5).float() == y).float().mean().item()
print("training accuracy:", round(accuracy, 3))

---
# Part 4 · Using a GPU

A **GPU** (graphics processing unit) can do huge numbers of multiplications at the same time, which makes training large networks much faster. In Google Colab you can switch one on with *Runtime → Change runtime type → GPU*. In PyTorch, you move the model and the data onto the GPU with `.to(device)`. The line below picks the GPU (`"cuda"`) if there is one, and the ordinary processor (`"cpu"`) otherwise, so the code works either way. Everything that is used together must be on the same device.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using", device)
model = LogisticRegression(2).to(device)
print(model(x2.to(device))[:3])

---
# Part 5 · A sentiment classifier in PyTorch

Now back to the Yelp reviews. This time the one-hot data has been prepared in advance and saved as a table in a CSV file. We read it with **pandas**, a Python library for working with tables of data. A pandas table is called a **DataFrame**. `df.iloc[rows, columns]` selects rows and columns by position, just like indexing a NumPy matrix, and `.values` turns the result into a NumPy array.

In this file, column 0 holds the review text, column 1 its label, and columns 2 to 5001 the one-hot encoding of the 5000 most frequent words. Reading it takes a little while.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60342/refs/heads/main/yelp_reviews_oh.csv.gz
!gunzip -f yelp_reviews_oh.csv.gz

In [ ]:
import pandas as pd
reviews_df = pd.read_csv("yelp_reviews_oh.csv")
print(reviews_df.shape)
print(reviews_df.iloc[:3, :5])          # the first 3 rows and the first 5 columns

### 🏋️ Exercise 4 — Preparing the tensors

1. `reviews_oh`: a float32 tensor of columns 2 to 5001 (`reviews_df.iloc[:, 2:5002].values`)
2. `labels`: a float32 tensor holding 1 where column 1 is `"positive"` and 0 elsewhere. *Hint:* `(reviews_df.iloc[:, 1] == "positive").values` gives an array of `True`/`False`.

The scikit-learn function `train_test_split` then splits both into training and test sets (20% for testing) in one go.

In [ ]:
from sklearn.model_selection import train_test_split

reviews_oh = ???
labels = ???
reviews_train, reviews_test, labels_train, labels_test = train_test_split(reviews_oh, labels, test_size=0.2, random_state=30)
print(reviews_train.shape, reviews_test.shape)

Here are two helper functions, given for you. `train` trains any binary classifier model on the training data with mini-batches, and returns the loss for each epoch. `precision_recall` calculates the precision and recall of a model on some data. Note the two new method calls:

* `model.train()` and `model.eval()` switch a model between training and evaluation mode. This matters for dropout, in Part 6.
* `with torch.no_grad():` tells PyTorch not to record calculations for gradients, which saves time and memory when we only want predictions.

In [ ]:
def train(model, X, y, n_epochs=20, batch_size=32, lr=0.1):
    """Train a binary classifier with mini-batches and return the loss for each epoch."""
    model.to(device)
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    epoch_loss = []
    for epoch in range(n_epochs):
        model.train()
        total = 0.0
        for s in gen_batches(len(X), batch_size):
            inputs, targets = X[s].to(device), y[s].to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(inputs).squeeze(1), targets)
            loss.backward()
            optimizer.step()
            total += loss.item()
        epoch_loss.append(total)
    return epoch_loss

def precision_recall(model, X, y):
    """Return the precision and recall of a binary classifier model on X and y."""
    model.eval()
    with torch.no_grad():
        pred = (model(X.to(device)).squeeze(1) > 0.5).int().cpu()
    tp = ((pred == 1) & (y == 1)).sum().item()
    fp = ((pred == 1) & (y == 0)).sum().item()
    fn = ((pred == 0) & (y == 1)).sum().item()
    return round(tp / (tp + fp), 3), round(tp / (tp + fn), 3)

### 🏋️ Exercise 5 — Train and test a logistic regression classifier

1. Create a `LogisticRegression` model for the 5000 one-hot features, and train it with `train` (the default settings are fine). Plot the loss.
2. Print its precision and recall on the **test** data and on the **training** data. Is there a sign of overfitting?

In [ ]:
torch.manual_seed(0)
lr_model = ???
losses = ???
plt.plot(losses)
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()
print("test precision and recall:", ???)
print("training precision and recall:", ???)

---
# Part 6 · Bigger networks and dropout

The great advantage of PyTorch is that we can make a model more complex just by adding layers, without working out any new gradients. Here is a network with one hidden layer of 250 units, using the ReLU activation from Week 9. The only rule is that each layer's number of inputs must match the previous layer's number of outputs.

In [ ]:
class MultilayerNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(5000, 250)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(250, 1)

    def forward(self, x):
        x = self.relu(self.layer_1(x))
        return torch.sigmoid(self.layer_2(x))

### 🏋️ Exercise 6 — A multilayer network

Train a `MultilayerNetwork` with `train` for 10 epochs, then print its precision and recall on the test and the training data. How does it compare with logistic regression?

In [ ]:
torch.manual_seed(0)
mlp = ???
losses = ???
print("test:", ???)
print("training:", ???)

### 🏋️ Exercise 7 — Two hidden layers

Complete `TwoHiddenLayers`, a network with **two** hidden layers: 5000 inputs → 250 units → 50 units → 1 output, with a ReLU after each hidden layer. Then train and test it as in Exercise 6.

In [ ]:
class TwoHiddenLayers(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(5000, 250)
        self.layer_2 = ???
        self.layer_3 = ???
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer_1(x))
        x = ???
        return ???

torch.manual_seed(0)
deep = TwoHiddenLayers()
losses = train(deep, reviews_train, labels_train, n_epochs=10)
print("test:", precision_recall(deep, reviews_test, labels_test))
print("training:", precision_recall(deep, reviews_train, labels_train))

### Dropout

The bigger a model gets, the more easily it **overfits**: it fits the training data almost perfectly but does less well on new data. One way of reducing overfitting is **dropout**. During training, at each step, a random proportion of the units in a layer is switched off (set to 0). This stops the network from relying too heavily on any single unit. When we evaluate the model, dropout is switched off again: that is what `model.eval()` does.

In PyTorch, `nn.Dropout(p)` switches off a proportion `p` of the values passing through it. Here it is applied to the hidden layer:

In [ ]:
class DropoutNetwork(nn.Module):
    def __init__(self, p_hidden=0.5):
        super().__init__()
        self.layer_1 = nn.Linear(5000, 250)
        self.relu = nn.ReLU()
        self.dropout_hidden = nn.Dropout(p_hidden)
        self.layer_2 = nn.Linear(250, 1)

    def forward(self, x):
        x = self.relu(self.layer_1(x))
        x = self.dropout_hidden(x)
        return torch.sigmoid(self.layer_2(x))

layer = nn.Dropout(0.5)
print(layer(torch.ones(10)))          # about half the values are switched off (the rest are doubled to compensate)

### 🏋️ Exercise 8 — Dropout on the inputs and the hidden layer

1. Complete `DropoutNetwork2`, which applies dropout to the **inputs** as well (with proportion `p_input`), before the first layer.
2. Train and test it with a few different dropout values, for example `p_input` of 0.2 or 0.5 and `p_hidden` of 0.5 or 0.8. What effect does dropout have on performance on the training data and on the test data?

In [ ]:
class DropoutNetwork2(nn.Module):
    def __init__(self, p_input=0.2, p_hidden=0.5):
        super().__init__()
        self.dropout_input = ???
        self.layer_1 = nn.Linear(5000, 250)
        self.relu = nn.ReLU()
        self.dropout_hidden = nn.Dropout(p_hidden)
        self.layer_2 = nn.Linear(250, 1)

    def forward(self, x):
        x = ???
        x = self.relu(self.layer_1(x))
        x = self.dropout_hidden(x)
        return torch.sigmoid(self.layer_2(x))

for p_input, p_hidden in [(0.2, 0.5), (0.5, 0.8)]:
    torch.manual_seed(0)
    net = DropoutNetwork2(p_input, p_hidden)
    train(net, reviews_train, labels_train, n_epochs=10)
    print(p_input, p_hidden, "test:", precision_recall(net, reviews_test, labels_test),
          "training:", precision_recall(net, reviews_train, labels_train))

---
## ✏️ Summary

This week you have:

* used PyTorch **tensors**, and let PyTorch work out gradients with `backward()`
* written a model as a class inheriting from `nn.Module`, with layers such as `nn.Linear`, `nn.ReLU` and `nn.Dropout`
* trained models with a **loss function** and an **optimiser** (`zero_grad`, `backward`, `step`), using mini-batches, on the CPU or a GPU
* built a sentiment classifier, made it deeper, and used **dropout** to reduce overfitting